In [11]:
from dotenv import load_dotenv
import requests
import json
from datetime import datetime
import use_case.Stock_Price_Prediction.predict.sentiment_predict as sentiment
from itertools import repeat
import pandas as pd
import time
import os

# Get Access Key

In [5]:
load_dotenv();

In [6]:
api_key = os.getenv("ALPHA_VANTAGE")

# Prepare paramters to get news

In [ ]:
parameters = {
    "topics" : ['earnings', 'financial_markets' , 'economy_fiscal', 'economy_monetary' , 'economy_macro', 'energy_transportation', 'finance'] ,   
    "ticker": "VOO",
    "time_from": "20250101T0000",
    "time_to" : "20251230T2359",   
}


# Move any list-valued items to the end (keep original order within each group)
parameters = {
        **{k: v for k, v in parameters.items() if not isinstance(v, list)},
        **{k: v for k, v in parameters.items() if isinstance(v, list)},
}

In [ ]:
standard = [f"{k}={v}" for k, v in parameters.items() if not isinstance(v, list)]
expanded = [(f"{k}={item}" , item ) for k, v in parameters.items() if isinstance(v, list) for item in v]

full_param_list = [("&".join([*standard, e]) ,t) for e, t in expanded]

# Function to get News

In [ ]:
def get_news(parameter):
    
    parameter = '&' + parameter  if parameter != '' and parameter[0] != '&' else parameter         
    
    url = f'https://www.alphavantage.co/query?limit=1000&apikey={api_key}&function=NEWS_SENTIMENT{parameter}'

    try:
        
        r = requests.get(url)
        data = r.json()   
    
        stock_news = [(item['time_published'][:8], item['summary'].strip()) for item in data['feed'] ]
    
        return stock_news

    except Exception as e:
       
        print(e)
        print(data)

# Get News List by topic

In [ ]:
full_news_list = []

for item, topic in full_param_list:

    time.sleep(0.5)
    
    news = get_news( item )

    
    if not news == None:

        full_news_list.append({"topic": topic,"news" : news })

# Save to text file

In [ ]:
# with open('news.txt', 'w') as f:
    
#     json.dump(full_news_list, f)    

In [ ]:
for item in full_news_list:
    
    topic = item['topic']
    newsList = item['news']

    # print('='*50,'\n', topic, ' :')
    try:
        for news in newsList[:2]:
            date = news[0]
            sentense = news[1]
    
            # print( '-'*5,date, '-'*40)
            # print(sentense)   
    
    except Exception as e:
        print(e)

# Get Data from txt file

In [7]:
with open('news.txt', 'r') as file:
    
    data = json.load(file)

In [ ]:
predict_result_list = [] 

for item in data:
    
    topic = item['topic']
    newsList = item['news'][:50]
    
    dateList = [n[0] for n in newsList]
    sentenseList = [n[1] for n in newsList]

    predicts_list, result,correct,incorrect = sentiment(sentenseList)    
    result = list(zip(dateList,sentenseList, predicts_list, repeat(topic)))
    
    result = [{"date":d,"topic":t,"sentense":s,"predict":p} for d,s,p,t in result]

    predict_result_list.extend(result)

In [19]:
df = pd.DataFrame(predict_result_list)

df['date'] = pd.to_datetime(df['date'], format='%Y%m%d')

In [21]:
pt = df.pivot_table(
    index='date',
    columns=['topic', 'predict'],
    aggfunc='size',
    fill_value=0
)

# flatten MultiIndex columns -> "topic_predict"
pt.columns = [f"{topic}_{pred}" for topic, pred in pt.columns]
pt = pt.reset_index()


,date,earnings_0,earnings_1,earnings_2,economy_fiscal_0,economy_fiscal_1,economy_fiscal_2,economy_macro_0,economy_macro_1,economy_macro_2,...,economy_monetary_2,energy_transportation_0,energy_transportation_1,energy_transportation_2,finance_0,finance_1,finance_2,financial_markets_0,financial_markets_1,financial_markets_2
0,2025-12-30,6,16,28,3,9,38,6,7,37,...,25,5,17,28,2,9,39,5,14,31


In [ ]:
df.to_csv('data_alphavantage', index = False)